In [19]:
import boto3
import datetime
import os
from datetime import timezone
import json

In [20]:
# --- CONFIGURATION ---
REGION = "us-east-1"
DRY_RUN = False  # Set to False to actually delete resources
SNAPSHOT_MAX_AGE_DAYS = 30

In [21]:
def load_env():
    """Simple manual loader for .env file if python-dotenv is not available."""
    env_vars = {}
    if os.path.exists(".env"):
        with open(".env") as f:
            for line in f:
                if "=" in line and not line.startswith("#"):
                    key, value = line.strip().split("=", 1)
                    # Remove quotes if present
                    env_vars[key.strip()] = value.strip().strip('"').strip("'")
    return env_vars

# Load credentials from .env
env = load_env()
session = boto3.Session(
    aws_access_key_id=env.get('aws_access_key_id'),
    aws_secret_access_key=env.get('aws_secret_access_key'),
    region_name=REGION
)
ec2 = session.client('ec2')

In [25]:
# print(ec2.describe_volumes()['Volumes'] )
# print(ec2.describe_instances()['Reservations'])
print(ec2.describe_addresses()['Addresses'])
# print(ec2.describe_snapshots(OwnerIds=['self'])['Snapshots'])

[{'AllocationId': 'eipalloc-02828e8637b64e57e', 'Domain': 'vpc', 'PublicIpv4Pool': 'amazon', 'NetworkBorderGroup': 'us-east-1', 'PublicIp': '107.21.232.58'}]


In [7]:
#ebs volume
volumes = ec2.describe_volumes()['Volumes']
for volume in volumes:
    volume_id = volume['VolumeId']
    volume_state = volume['State']
    volume_name = next((tag['Value'] for tag in volume.get('Tags', []) if tag['Key'] == 'Name'), 'N/A')
    print(f"Volume ID: {volume_id}, State: {volume_state}, Name: {volume_name}")

Volume ID: vol-03c99d0dcb7b2aa5c, State: in-use, Name: N/A
Volume ID: vol-033ed05ba79f7716d, State: in-use, Name: N/A
Volume ID: vol-0b1bbe4fff65f2b22, State: in-use, Name: N/A
Volume ID: vol-0a484ae9309a252c5, State: in-use, Name: N/A
Volume ID: vol-01f41d15c250b71cd, State: available, Name: umused-2
Volume ID: vol-0b876b9f6e2fc7b03, State: available, Name: unused-1


In [8]:
#list of ec2 instances
instances = ec2.describe_instances()['Reservations']
for reservation in instances:
    for instance in reservation['Instances']:
        instance_id = instance['InstanceId']
        instance_state = instance['State']['Name']
        instance_name = next((tag['Value'] for tag in instance.get('Tags', []) if tag['Key'] == 'Name'), 'N/A')
        print(f"Instance ID: {instance_id}, State: {instance_state}, Name: {instance_name}")

Instance ID: i-02b21c3ca1a3cb2fb, State: running, Name: test1
Instance ID: i-0a0f33723cd31d786, State: running, Name: test2
Instance ID: i-05e2cd7cb20c61d73, State: running, Name: test3
Instance ID: i-0da0f10deb7898a1a, State: running, Name: test4


In [11]:
#list of unused elastic ips
addresses = ec2.describe_addresses()['Addresses']
for address in addresses:
    allocation_id = address.get('AllocationId')
    print(f"Elastic IP Allocation ID: {allocation_id}")
    instance_id = address.get('InstanceId')
    if not instance_id:
        print(f"Elastic IP {allocation_id} is not associated with any instance.")

In [14]:
# list of Old Snapshots
snapshots = ec2.describe_snapshots(OwnerIds=['self'])['Snapshots']
now = datetime.datetime.now(timezone.utc)
for snapshot in snapshots:
    snapshot_id = snapshot['SnapshotId']
    print(f"Snapshot ID: {snapshot_id}")
    start_time = snapshot['StartTime']
    age_days = (now - start_time).days
    if age_days > SNAPSHOT_MAX_AGE_DAYS:
        print(f"Snapshot {snapshot_id} is {age_days} days old and may be a candidate for deletion.")

Snapshot ID: snap-0d522e14f23d96b59


In [15]:
#delete stopped instances
for reservation in instances:
    for instance in reservation['Instances']:
        instance_id = instance['InstanceId']
        instance_state = instance['State']['Name']
        if instance_state == 'stopped':
            print(f"Instance {instance_id} is stopped and will be terminated.")
            if not DRY_RUN:
                # ec2.terminate_instances(InstanceIds=[instance_id])     

SyntaxError: incomplete input (2489352881.py, line 9)

In [16]:
#delete unused ebs volumes
for volume in volumes: 
    volume_id = volume['VolumeId'] 
    volume_state = volume['State'] 
    if volume_state == 'available': 
        print(f"Volume {volume_id} is available and will be deleted.") 
        if not DRY_RUN: 
            print("deleting volume...")
            # ec2.delete_volume(VolumeId=volume_id)

Volume vol-01f41d15c250b71cd is available and will be deleted.
deleting volume...
Volume vol-0b876b9f6e2fc7b03 is available and will be deleted.
deleting volume...


In [17]:
#delete unused elastic ips
addresses = ec2.describe_addresses()['Addresses']
for address in addresses:    
    allocation_id = address.get('AllocationId') 
    instance_id = address.get('InstanceId') 
    if not instance_id: 
        print(f"Elastic IP {allocation_id} is not associated with any instance and will be released.") 
        if not DRY_RUN: 
            print("deleting elastic ip...")
            # ec2.release_address(AllocationId=allocation_id) 

In [ ]:
#delete snapshots older than  30 days
snapshots = ec2.describe_snapshots(OwnerIds=['self'])['Snapshots']
now = datetime.datetime.now(timezone.utc)
for snapshot in snapshots:
    snapshot_id = snapshot['SnapshotId']
    start_time = snapshot['StartTime']
    age_days = (now - start_time).days
    if age_days > SNAPSHOT_MAX_AGE_DAYS:
        print(f"Snapshot {snapshot_id} is {age_days} days old and will be deleted.")
        if not DRY_RUN:
            print("deleting snapshot...")
            # ec2.delete_snapshot(SnapshotId=snapshot_id)